# Road Damage Detection — 01. Train YOLOv8s

Complete workflow for **YOLOv8** with `yolov8s.pt`:

- fair shared baseline;
- validation-only evaluation;
- optional equal-budget hyperparameter tuning;
- final tuned retraining;
- prediction preview and persistent metadata.

The test split is intentionally not used here. Final test evaluation happens once in notebook 04.

## Fair-comparison protocol

Keep these identical across the three model notebooks: `s` size tier, image size 640,
batch 16, AdamW, 100 epochs, patience 20, seed 42, augmentations, dataset split, and GPU type.
If batch 16 causes OOM, change it to 8 in **all three** notebooks.

In [ ]:
# Versions pinned for reproducibility and YOLO26 support.
%pip install -q \
    ultralytics==8.4.92 \
    roboflow==1.3.13 \
    pyyaml==6.0.3 \
    pandas>=2.2,<3 \
    matplotlib>=3.9,<4 \
    pillow>=11,<13 \
    psutil>=6,<8 \
    onnx>=1.17,<2 \
    onnxruntime>=1.20,<2

In [ ]:
from __future__ import annotations

import gc
import json
import os
import platform
import random
import shutil
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from PIL import Image

SEED = 42

def seed_everything(seed: int = SEED) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

print('Python:', sys.version)
print('Platform:', platform.platform())
print('PyTorch:', torch.__version__)
print('Ultralytics:', ultralytics.__version__)
print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA runtime:', torch.version.cuda)
else:
    raise RuntimeError(
        'No GPU detected. In Colab choose Runtime → Change runtime type → T4 GPU.'
    )

## Dataset preflight

This notebook uses only the processed object-detection dataset generated by notebook 00:

```text
/content/drive/MyDrive/RoadDamageYOLO/data/road-damage-detection-bbox-v1
```

It copies the dataset to Colab's local disk and creates a runtime-specific YAML that points to:

```text
/content/road-damage-detection-bbox-v1
```

Do not change these paths back to `road-damage-1`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/RoadDamageYOLO")

# This is the processed object-detection dataset produced by notebook 00.
PERSISTENT_DATASET_DIR = (
    PROJECT_ROOT
    / "data"
    / "road-damage-detection-bbox-v1"
)

# Training from Colab's local disk is faster than reading thousands of files
# directly from Google Drive.
LOCAL_DATASET_DIR = Path(
    "/content/road-damage-detection-bbox-v1"
)

RUNS_ROOT = PROJECT_ROOT / "runs"
REPORTS_ROOT = PROJECT_ROOT / "reports"
EXPORTS_ROOT = PROJECT_ROOT / "exports"

for directory in [
    RUNS_ROOT,
    REPORTS_ROOT,
    EXPORTS_ROOT,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Persistent dataset:", PERSISTENT_DATASET_DIR)
print("Local dataset:", LOCAL_DATASET_DIR)

In [ ]:
def find_validation_folder(dataset_dir: Path) -> str:
    if (dataset_dir / "valid").exists():
        return "valid"
    if (dataset_dir / "val").exists():
        return "val"
    raise FileNotFoundError(
        f"Neither 'valid' nor 'val' exists inside {dataset_dir}."
    )


def verify_dataset_structure(dataset_dir: Path) -> dict[str, int]:
    validation_folder = find_validation_folder(dataset_dir)

    split_folders = {
        "train": "train",
        "val": validation_folder,
        "test": "test",
    }

    counts = {}
    missing_paths = []

    for split_name, folder_name in split_folders.items():
        images_dir = dataset_dir / folder_name / "images"
        labels_dir = dataset_dir / folder_name / "labels"

        if not images_dir.exists():
            missing_paths.append(str(images_dir))
        if not labels_dir.exists():
            missing_paths.append(str(labels_dir))

        if images_dir.exists():
            counts[split_name] = sum(
                1
                for path in images_dir.iterdir()
                if path.is_file()
                and path.suffix.lower()
                in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
            )

    if missing_paths:
        raise FileNotFoundError(
            "The processed dataset is incomplete. Missing:\n"
            + "\n".join(missing_paths)
        )

    if not all(counts.get(split, 0) > 0 for split in ["train", "val", "test"]):
        raise RuntimeError(
            f"One or more dataset splits contain no images: {counts}"
        )

    return counts


def copy_dataset_to_local(
    persistent_dir: Path = PERSISTENT_DATASET_DIR,
    local_dir: Path = LOCAL_DATASET_DIR,
    force_refresh: bool = False,
) -> None:
    if not persistent_dir.exists():
        raise FileNotFoundError(
            f"Processed dataset not found at:\n{persistent_dir}\n\n"
            "Run 00_Data_Setup_and_Audit_UPDATED.ipynb first."
        )

    persistent_counts = verify_dataset_structure(persistent_dir)
    print("Persistent dataset image counts:", persistent_counts)

    local_is_ready = False
    if local_dir.exists() and not force_refresh:
        try:
            local_counts = verify_dataset_structure(local_dir)
            local_is_ready = local_counts == persistent_counts
            if local_is_ready:
                print("Using the existing verified local dataset copy.")
        except Exception:
            local_is_ready = False

    if not local_is_ready:
        if local_dir.exists():
            print("Removing stale or incomplete local dataset copy...")
            shutil.rmtree(local_dir)

        print("Copying processed dataset from Google Drive to Colab local storage...")
        shutil.copytree(persistent_dir, local_dir)

        local_counts = verify_dataset_structure(local_dir)
        if local_counts != persistent_counts:
            raise RuntimeError(
                "Local copy verification failed.\n"
                f"Persistent counts: {persistent_counts}\n"
                f"Local counts: {local_counts}"
            )

    print("Local dataset ready:", local_dir)


def build_local_data_yaml(
    dataset_dir: Path = LOCAL_DATASET_DIR,
) -> Path:
    validation_folder = find_validation_folder(dataset_dir)

    yaml_candidates = [
        dataset_dir / "data_detection_drive.yaml",
        dataset_dir / "data_detection.yaml",
        dataset_dir / "data.yaml",
    ]

    source_yaml = next(
        (path for path in yaml_candidates if path.exists()),
        None,
    )

    if source_yaml is None:
        raise FileNotFoundError(
            f"No dataset YAML was found inside {dataset_dir}."
        )

    with source_yaml.open("r", encoding="utf-8") as file:
        source_config = yaml.safe_load(file)

    names = source_config.get("names")
    if not names:
        raise ValueError(
            f"The dataset YAML {source_yaml} does not contain class names."
        )

    local_config = {
        "path": str(dataset_dir.resolve()),
        "train": "train/images",
        "val": f"{validation_folder}/images",
        "test": "test/images",
        "names": names,
    }

    output_yaml = dataset_dir / "data_colab.yaml"
    with output_yaml.open("w", encoding="utf-8") as file:
        yaml.safe_dump(
            local_config,
            file,
            sort_keys=False,
            allow_unicode=True,
        )

    return output_yaml


# Set to True only when the local copy is stale or after changing dataset version.
FORCE_DATASET_REFRESH = False

copy_dataset_to_local(force_refresh=FORCE_DATASET_REFRESH)
DATA_YAML = build_local_data_yaml()

image_counts = verify_dataset_structure(LOCAL_DATASET_DIR)

print("\nVerified local dataset counts:")
for split, count in image_counts.items():
    print(f"  {split}: {count}")

print("\nTraining/evaluation YAML:")
print(DATA_YAML.read_text())

assert str(LOCAL_DATASET_DIR) in DATA_YAML.read_text(), (
    "The local dataset YAML does not point to the local processed dataset."
)

In [ ]:
environment = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda_runtime': torch.version.cuda,
    'cuda_available': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'ultralytics': ultralytics.__version__,
}
(REPORTS_ROOT / 'environment.json').write_text(
    json.dumps(environment, indent=2), encoding='utf-8'
)
environment

## 1. Experiment configuration

In [ ]:
from ultralytics import YOLO

MODEL_FAMILY = 'YOLOv8'
MODEL_WEIGHTS = 'yolov8s.pt'
MODEL_TAG = 'YOLOv8s'

DEVICE = 0
IMGSZ = 640
BATCH = 16
EPOCHS = 100
PATIENCE = 20
WORKERS = 2

MODEL_RUNS_DIR = RUNS_ROOT / MODEL_TAG
MODEL_RUNS_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_NAME = 'baseline_seed42'
FINAL_NAME = 'final_tuned_seed42'
TUNE_NAME = 'tune_equal_budget'
BASELINE_DIR = MODEL_RUNS_DIR / BASELINE_NAME
FINAL_DIR = MODEL_RUNS_DIR / FINAL_NAME

RUN_BASELINE = True
FORCE_BASELINE_RETRAIN = False

# Tuning is expensive and is disabled to prevent accidental execution.
RUN_TUNING = False
RESUME_TUNING = False
TUNE_ITERATIONS = 20
TUNE_EPOCHS = 40

# Enable only after a completed tuning study.
RUN_FINAL_TRAIN = False
FORCE_FINAL_RETRAIN = False

print('Model:', MODEL_WEIGHTS)
print('Run directory:', MODEL_RUNS_DIR)

## 2. Shared baseline configuration

In [ ]:
COMMON_TRAIN_ARGS = {
    'data': str(DATA_YAML), 'imgsz': IMGSZ, 'epochs': EPOCHS, 'batch': BATCH,
    'patience': PATIENCE, 'device': DEVICE, 'workers': WORKERS,
    'seed': SEED, 'deterministic': True, 'amp': True,
    'optimizer': 'AdamW', 'lr0': 0.001, 'lrf': 0.01,
    'weight_decay': 0.0005, 'warmup_epochs': 3.0, 'cos_lr': True,
    'close_mosaic': 10, 'hsv_h': 0.015, 'hsv_s': 0.50, 'hsv_v': 0.40,
    'degrees': 3.0, 'translate': 0.10, 'scale': 0.40, 'shear': 1.0,
    'perspective': 0.0005, 'flipud': 0.0, 'fliplr': 0.5,
    'mosaic': 0.8, 'mixup': 0.05, 'cache': False,
    'plots': True, 'save': True, 'save_period': 10, 'verbose': True,
}
(MODEL_RUNS_DIR / 'baseline_config.json').write_text(
    json.dumps(COMMON_TRAIN_ARGS, indent=2), encoding='utf-8'
)
display(pd.Series(COMMON_TRAIN_ARGS, name='value'))

## 3. Train or resume the fair baseline

In [ ]:
baseline_best = BASELINE_DIR / 'weights' / 'best.pt'
baseline_last = BASELINE_DIR / 'weights' / 'last.pt'

if baseline_best.exists() and not FORCE_BASELINE_RETRAIN:
    print('Baseline exists; skipping:', baseline_best)
elif RUN_BASELINE:
    if baseline_last.exists() and not FORCE_BASELINE_RETRAIN:
        print('Resuming:', baseline_last)
        YOLO(str(baseline_last)).train(resume=True)
    else:
        YOLO(MODEL_WEIGHTS).train(
            **COMMON_TRAIN_ARGS,
            project=str(MODEL_RUNS_DIR),
            name=BASELINE_NAME,
            exist_ok=False,
        )
else:
    print('RUN_BASELINE=False')

print('Baseline checkpoint:', baseline_best if baseline_best.exists() else 'not available')

## 4. Validate the baseline on the validation split

In [ ]:
def metrics_to_dict(metrics, checkpoint: Path, stage: str) -> dict:
    precision = float(metrics.box.mp)
    recall = float(metrics.box.mr)
    loaded = YOLO(str(checkpoint))
    names = loaded.names if isinstance(loaded.names, dict) else dict(enumerate(loaded.names))
    maps = [float(value) for value in metrics.box.maps]
    return {
        'model_family': MODEL_FAMILY,
        'model_tag': MODEL_TAG,
        'stage': stage,
        'checkpoint': str(checkpoint),
        'precision': precision,
        'recall': recall,
        'f1': 2 * precision * recall / (precision + recall + 1e-12),
        'map50': float(metrics.box.map50),
        'map50_95': float(metrics.box.map),
        'per_class_map50_95': {
            str(names.get(index, index)): maps[index]
            for index in range(min(len(maps), len(names)))
        },
        'model_size_mb': checkpoint.stat().st_size / (1024 ** 2),
        'parameters': int(sum(parameter.numel() for parameter in loaded.model.parameters())),
        'speed_ms': {key: float(value) for key, value in metrics.speed.items()},
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
    }

if baseline_best.exists():
    metrics = YOLO(str(baseline_best)).val(
        data=str(DATA_YAML), split='val', imgsz=IMGSZ, batch=BATCH,
        device=DEVICE, workers=WORKERS, plots=True,
        project=str(MODEL_RUNS_DIR), name='baseline_validation',
    )
    baseline_summary = metrics_to_dict(metrics, baseline_best, 'baseline_validation')
    (MODEL_RUNS_DIR / 'baseline_validation_summary.json').write_text(
        json.dumps(baseline_summary, indent=2), encoding='utf-8'
    )
    display(pd.Series(baseline_summary))
else:
    print('No baseline checkpoint found.')

## 5. Equal-budget hyperparameter tuning

Set `RUN_TUNING=True` in the configuration cell. Defaults allocate 20 trials × 40 epochs
to each model. One iteration is one independent training run. Use `RESUME_TUNING=True`
after an interrupted session.

In [ ]:
SEARCH_SPACE = {
    'lr0': (1e-4, 5e-3), 'lrf': (0.01, 0.20),
    'weight_decay': (1e-5, 1e-3), 'warmup_epochs': (0.0, 5.0),
    'box': (5.0, 12.0), 'cls': (0.2, 1.5),
    'hsv_h': (0.0, 0.04), 'hsv_s': (0.20, 0.80), 'hsv_v': (0.20, 0.70),
    'degrees': (0.0, 7.0), 'translate': (0.0, 0.20), 'scale': (0.10, 0.60),
    'shear': (0.0, 3.0), 'perspective': (0.0, 0.001),
    'fliplr': (0.0, 0.50), 'mosaic': (0.40, 1.00),
    'mixup': (0.0, 0.15), 'close_mosaic': (5.0, 20.0),
}
display(pd.Series(SEARCH_SPACE))

In [ ]:
if RUN_TUNING:
    YOLO(MODEL_WEIGHTS).tune(
        data=str(DATA_YAML), imgsz=IMGSZ, batch=BATCH,
        epochs=TUNE_EPOCHS, iterations=TUNE_ITERATIONS,
        optimizer='AdamW', device=DEVICE, workers=WORKERS,
        seed=SEED, deterministic=True, amp=True,
        space=SEARCH_SPACE, project=str(MODEL_RUNS_DIR), name=TUNE_NAME,
        resume=RESUME_TUNING, plots=True, save=True, val=True,
    )
else:
    print('Tuning disabled. Set RUN_TUNING=True when ready.')

## 6. Load the best hyperparameters

In [ ]:
def newest_file(root: Path, filename: str) -> Path | None:
    candidates = list(root.rglob(filename))
    return max(candidates, key=lambda path: path.stat().st_mtime) if candidates else None

best_hypers_path = newest_file(MODEL_RUNS_DIR, 'best_hyperparameters.yaml')
best_hypers = {}
if best_hypers_path:
    with best_hypers_path.open('r', encoding='utf-8') as file:
        best_hypers = yaml.safe_load(file) or {}
    print(best_hypers_path)
    display(pd.Series(best_hypers, name='value'))
else:
    print('No tuning result found yet.')

## 7. Retrain from pretrained weights using the tuned configuration

In [ ]:
final_best = FINAL_DIR / 'weights' / 'best.pt'
final_last = FINAL_DIR / 'weights' / 'last.pt'
filtered_hypers = {key: value for key, value in best_hypers.items() if key in SEARCH_SPACE}

if final_best.exists() and not FORCE_FINAL_RETRAIN:
    print('Final tuned checkpoint exists:', final_best)
elif RUN_FINAL_TRAIN:
    if not filtered_hypers:
        raise RuntimeError('Complete tuning before final retraining.')
    if final_last.exists() and not FORCE_FINAL_RETRAIN:
        YOLO(str(final_last)).train(resume=True)
    else:
        final_args = dict(COMMON_TRAIN_ARGS)
        final_args.update(filtered_hypers)
        (MODEL_RUNS_DIR / 'final_tuned_config.json').write_text(
            json.dumps(final_args, indent=2), encoding='utf-8'
        )
        YOLO(MODEL_WEIGHTS).train(
            **final_args, project=str(MODEL_RUNS_DIR),
            name=FINAL_NAME, exist_ok=False,
        )
else:
    print('Final training disabled. Set RUN_FINAL_TRAIN=True after tuning.')

## 8. Validate the final tuned model on validation data

In [ ]:
if final_best.exists():
    metrics = YOLO(str(final_best)).val(
        data=str(DATA_YAML), split='val', imgsz=IMGSZ, batch=BATCH,
        device=DEVICE, workers=WORKERS, plots=True,
        project=str(MODEL_RUNS_DIR), name='final_tuned_validation',
    )
    tuned_summary = metrics_to_dict(metrics, final_best, 'final_tuned_validation')
    (MODEL_RUNS_DIR / 'final_tuned_validation_summary.json').write_text(
        json.dumps(tuned_summary, indent=2), encoding='utf-8'
    )
    display(pd.Series(tuned_summary))
else:
    print('No final tuned checkpoint exists yet.')

## 9. Prediction preview

In [ ]:
import matplotlib.pyplot as plt
selected = final_best if final_best.exists() else baseline_best
if selected.exists():
    valid_name = 'valid' if (LOCAL_DATASET_DIR / 'valid').exists() else 'val'
    images = sorted(
        path for path in (LOCAL_DATASET_DIR / valid_name / 'images').iterdir()
        if path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}
    )
    image_path = random.Random(SEED).choice(images)
    result = YOLO(str(selected)).predict(
        source=str(image_path), imgsz=IMGSZ, conf=0.25,
        device=DEVICE, verbose=False,
    )[0]
    plt.figure(figsize=(14, 8))
    plt.imshow(result.plot()[..., ::-1])
    plt.title(f'{MODEL_TAG} prediction — {image_path.name}')
    plt.axis('off')
    plt.show()
    print('Detections:', len(result.boxes))
else:
    print('Train a model first.')

## 10. Save selected checkpoint metadata

In [ ]:
selected = final_best if final_best.exists() else baseline_best
if selected.exists():
    metadata = {
        'model_family': MODEL_FAMILY, 'model_tag': MODEL_TAG,
        'weights_source': MODEL_WEIGHTS, 'selected_checkpoint': str(selected),
        'selection_stage': 'final_tuned' if final_best.exists() else 'baseline',
        'dataset_yaml': str(DATA_YAML), 'seed': SEED, 'imgsz': IMGSZ,
        'batch': BATCH, 'ultralytics_version': ultralytics.__version__,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
    }
    (MODEL_RUNS_DIR / 'selected_checkpoint.json').write_text(
        json.dumps(metadata, indent=2), encoding='utf-8'
    )
    print(json.dumps(metadata, indent=2))

## Completed

Continue with the next model notebook. Notebook 04 performs the untouched test evaluation,
matched-box IoU analysis, latency/FPS benchmark, size and VRAM comparison, and ONNX export.

### Shared artifact contract

This notebook writes checkpoints and metrics to Google Drive. The later comparison
notebook discovers them through the fixed run-directory names. Do not manually rename:

- `YOLOv8s`, `YOLO11s`, or `YOLO26s`
- `baseline_seed42`
- `final_tuned_seed42`
- their `weights/best.pt` files